# From characters to Transformers — one executable `aabid` thread

This CPU-only PyTorch notebook follows the same held example as the mega-lecture. We keep the learning objective fixed—predict the next token—and replace the context machinery step by step:

`fixed-window MLP → single-query attention → self-attention → causal decoder block`.

Every section includes small shape or numerical assertions. There are no downloads and the two tiny models train only long enough to overfit the six teaching examples.

In [ ]:
import math
import random

import torch
from torch import nn
import torch.nn.functional as F

SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_printoptions(precision=4, sci_mode=False)
device = torch.device('cpu')
print(f'PyTorch {torch.__version__} · device={device} · seed={SEED}')

## 1 · `aabid` gives exactly six supervised examples

The vocabulary is `-` plus the 26 lowercase letters. The toy boundary token supplies left padding and the end-of-sequence target. With context length three, the dataset has six rows.

In [ ]:
vocab = ['-'] + list('abcdefghijklmnopqrstuvwxyz')
stoi = {token: i for i, token in enumerate(vocab)}
itos = {i: token for token, i in stoi.items()}

pairs = [
    ('---', 'a'),
    ('--a', 'a'),
    ('-aa', 'b'),
    ('aab', 'i'),
    ('abi', 'd'),
    ('bid', '-'),
]
X = torch.tensor([[stoi[c] for c in context] for context, _ in pairs], dtype=torch.long)
y = torch.tensor([stoi[target] for _, target in pairs], dtype=torch.long)

assert len(vocab) == 27
assert len(pairs) == 6
assert X.shape == (6, 3) and y.shape == (6,)
assert pairs == [('---', 'a'), ('--a', 'a'), ('-aa', 'b'), ('aab', 'i'), ('abi', 'd'), ('bid', '-')]
for i, (context, target) in enumerate(pairs, start=1):
    print(f'{i}: {context} → {target}')

## 2 · Embedding lookup + concatenate + fixed-context MLP

Each row of `X` selects three learned embedding vectors. Concatenation preserves their slots, giving a fixed `3 × d_embed` input to an ordinary classifier with 27 logits.

In [ ]:
class FixedWindowMLP(nn.Module):
    def __init__(self, vocab_size=27, context_len=3, d_embed=4, hidden=32):
        super().__init__()
        self.context_len = context_len
        self.d_embed = d_embed
        self.embedding = nn.Embedding(vocab_size, d_embed)
        self.classifier = nn.Sequential(
            nn.Linear(context_len * d_embed, hidden),
            nn.Tanh(),
            nn.Linear(hidden, vocab_size),
        )

    def forward(self, ids):
        embeddings = self.embedding(ids)
        logits = self.classifier(embeddings.flatten(start_dim=1))
        return logits, embeddings

torch.manual_seed(SEED)
mlp = FixedWindowMLP().to(device)
optimizer = torch.optim.AdamW(mlp.parameters(), lr=0.05, weight_decay=0.0)
for step in range(201):
    logits, embeddings = mlp(X)
    loss = F.cross_entropy(logits, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step in {0, 25, 100, 200}:
        print(f'step {step:3d} · loss {loss.item():.6f}')

with torch.no_grad():
    logits, embeddings = mlp(X)
    predictions = logits.argmax(dim=-1)
assert embeddings.shape == (6, 3, 4)
assert logits.shape == (6, 27)
assert torch.equal(predictions, y)
assert F.cross_entropy(logits, y).item() < 0.02
print('predictions:', [itos[i.item()] for i in predictions])

## 3 · Exact single-query attention for `a a b → i`

Use the lecture's two-dimensional vectors exactly. The query describes what the final `b` seeks; keys advertise matches; values carry the retrieved payload. We deliberately omit the square-root scale in this one-query arithmetic so the scores are exactly `[0, 0, 1]`.

In [ ]:
q_b = torch.tensor([0.0, 1.0])
K_exact = torch.tensor([[1.0, 0.0], [1.0, 0.0], [0.0, 1.0]])
V_exact = torch.tensor([[2.0, 0.0], [2.0, 0.0], [0.0, 2.0]])

scores_exact = K_exact @ q_b
weights_exact = torch.softmax(scores_exact, dim=0)
output_exact = weights_exact @ V_exact

expected_weights = torch.tensor([0.21194156, 0.21194156, 0.57611688])
expected_output = torch.tensor([0.84776622, 1.15223376])
assert torch.allclose(scores_exact, torch.tensor([0.0, 0.0, 1.0]))
assert torch.allclose(weights_exact, expected_weights, atol=1e-6)
assert torch.allclose(weights_exact.sum(), torch.tensor(1.0))
assert torch.allclose(output_exact, expected_output, atol=1e-6)
print('scores :', scores_exact)
print('weights:', weights_exact)
print('output  :', output_exact)

## 4 · Every row retrieves: row-wise self-attention

Rows are queries and columns are candidate source keys. `softmax(..., dim=-1)` therefore normalizes each query row—not each column.

In [ ]:
X_sa = torch.tensor([[1.0, 0.0], [1.0, 0.0], [0.0, 1.0]])
Q_sa = K_sa = V_sa = X_sa
scores_sa = Q_sa @ K_sa.T / math.sqrt(Q_sa.shape[-1])
A_sa = torch.softmax(scores_sa, dim=-1)
O_sa = A_sa @ V_sa

assert scores_sa.shape == (3, 3)
assert A_sa.shape == (3, 3) and O_sa.shape == (3, 2)
assert torch.allclose(A_sa.sum(dim=-1), torch.ones(3))
assert torch.all(A_sa >= 0)
print('row-wise attention weights:')
print(A_sa)
print('contextual outputs:')
print(O_sa)

## 5 · Causal masking makes future probability exactly zero

Mask the strict upper triangle before the row-wise softmax. Query row `t` may use keys `0…t`, and every future column must receive zero probability.

In [ ]:
X_causal = torch.tensor([[1.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
T = X_causal.shape[0]
scores_causal = X_causal @ X_causal.T / math.sqrt(X_causal.shape[-1])
future_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
masked_scores = scores_causal.masked_fill(future_mask, float('-inf'))
A_causal = torch.softmax(masked_scores, dim=-1)

assert torch.allclose(A_causal.sum(dim=-1), torch.ones(T))
assert torch.count_nonzero(torch.triu(A_causal, diagonal=1)) == 0
assert torch.isneginf(masked_scores[future_mask]).all()
print('future mask (1 = forbidden):')
print(future_mask.int())
print('causal attention weights:')
print(A_causal)

## 6 · One tiny pre-norm decoder block

The block has the modern teaching order: `LayerNorm → masked MHA → residual`, then `LayerNorm → MLP → residual`. Token and learned position embeddings provide the initial stream.

In [ ]:
class PreNormDecoderBlock(nn.Module):
    def __init__(self, d_model=24, n_heads=4, mlp_mult=2):
        super().__init__()
        assert d_model % n_heads == 0
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=0.0, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, mlp_mult * d_model),
            nn.GELU(),
            nn.Linear(mlp_mult * d_model, d_model),
        )

    def forward(self, x):
        _, T, _ = x.shape
        future_mask = torch.triu(
            torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1
        )
        h = self.ln1(x)
        attn_out, attn_weights = self.attn(
            h, h, h, attn_mask=future_mask, need_weights=True, average_attn_weights=False
        )
        x = x + attn_out
        x = x + self.ff(self.ln2(x))
        return x, attn_weights

class TinyDecoderLM(nn.Module):
    def __init__(self, vocab_size=27, d_model=24, n_heads=4, max_len=16):
        super().__init__()
        self.max_len = max_len
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.block = PreNormDecoderBlock(d_model, n_heads)
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, ids):
        B, T = ids.shape
        assert T <= self.max_len
        positions = torch.arange(T, device=ids.device)
        x = self.token_embedding(ids) + self.position_embedding(positions)[None, :, :]
        x, attn_weights = self.block(x)
        logits = self.lm_head(self.final_norm(x))
        return logits, attn_weights

## 7 · One causal pass, six shifted cross-entropy terms

Use the nine-token stream `---aabid-`. The first eight tokens are model inputs; the last eight are next-token targets. We score positions 2–7, which reproduce the same six targets as the fixed-window dataset.

In [ ]:
stream = '---aabid-'
stream_ids = torch.tensor([stoi[c] for c in stream], dtype=torch.long)
decoder_inputs = stream_ids[:-1].unsqueeze(0)
decoder_targets = stream_ids[1:].unsqueeze(0)
scored_positions = slice(2, None)

torch.manual_seed(SEED)
decoder = TinyDecoderLM().to(device)
logits, attn_weights = decoder(decoder_inputs)
initial_loss = F.cross_entropy(
    logits[:, scored_positions, :].reshape(-1, len(vocab)),
    decoder_targets[:, scored_positions].reshape(-1),
)
assert decoder_inputs.shape == decoder_targets.shape == (1, 8)
assert logits.shape == (1, 8, 27)
assert attn_weights.shape == (1, 4, 8, 8)
assert torch.count_nonzero(torch.triu(attn_weights[0], diagonal=1)) == 0
print(f'initial shifted CE: {initial_loss.item():.6f}')
for t in range(2, decoder_inputs.shape[1]):
    context = stream[t-2:t+1]
    target = stream[t+1]
    print(f'position {t}: {context} → {target}')

# Causality check: changing the last input cannot alter any earlier logit.
changed_inputs = decoder_inputs.clone()
changed_inputs[0, -1] = stoi['z']
changed_logits, _ = decoder(changed_inputs)
assert torch.allclose(logits[:, :-1], changed_logits[:, :-1], atol=1e-6)

optimizer = torch.optim.AdamW(decoder.parameters(), lr=0.03, weight_decay=0.0)
for step in range(301):
    logits, _ = decoder(decoder_inputs)
    loss = F.cross_entropy(
        logits[:, scored_positions, :].reshape(-1, len(vocab)),
        decoder_targets[:, scored_positions].reshape(-1),
    )
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step in {0, 25, 100, 300}:
        print(f'step {step:3d} · shifted CE {loss.item():.6f}')

with torch.no_grad():
    logits, _ = decoder(decoder_inputs)
    predicted_targets = logits[:, scored_positions, :].argmax(dim=-1)
    final_loss = F.cross_entropy(
        logits[:, scored_positions, :].reshape(-1, len(vocab)),
        decoder_targets[:, scored_positions].reshape(-1),
    )
assert final_loss.item() < 0.02
assert torch.equal(predicted_targets, decoder_targets[:, scored_positions])
print('predicted six targets:', ''.join(itos[i.item()] for i in predicted_targets[0]))

## 8 · Autoregressive generation closes the loop

Start from `---`, read the last-position distribution, append one chosen token, and repeat until the boundary token appears. Greedy decoding is deterministic here so the final assertion stays reproducible.

In [ ]:
@torch.no_grad()
def generate(model, prefix='---', max_new_tokens=12, temperature=1.0, greedy=True):
    model.eval()
    ids = torch.tensor([[stoi[c] for c in prefix]], dtype=torch.long)
    for _ in range(max_new_tokens):
        logits, _ = model(ids[:, -model.max_len:])
        next_logits = logits[:, -1, :] / temperature
        if greedy:
            next_id = next_logits.argmax(dim=-1, keepdim=True)
        else:
            next_id = torch.multinomial(torch.softmax(next_logits, dim=-1), 1)
        ids = torch.cat([ids, next_id], dim=1)
        if next_id.item() == stoi['-'] and ids.shape[1] > len(prefix):
            break
    return ''.join(itos[i.item()] for i in ids[0])

generated_stream = generate(decoder, prefix='---', greedy=True)
generated_name = generated_stream[3:-1]
assert generated_stream == '---aabid-'
assert generated_name == 'aabid'
print('generated stream:', generated_stream)
print('generated name  :', generated_name)

## Takeaway

The six targets and shifted cross-entropy objective never changed. What changed was the representation of previous tokens: fixed concatenation became query-dependent retrieval, then causal multi-head self-attention inside a residual decoder block.